In [3]:
import numpy as np

### Base class for agents. 

All agents should inherit from this class and implement the __call__ method.

In [4]:

class agent:
    # def Python mein kisi bhi function ko define karne (banane) ka keyword hai
    # __init__ "initialize" ka short form hai. Iske aage aur peeche double underscores (__) hote hain, jo Python ko batate hain ki yeh ek special (magic) method hai
    def __init__(self, name):
        self.name = name

    def name(self):
        return self.name

    def __call__(self, agent_history, opponent_history, burn_in=False):
        # NumPy being used
        return np.random.rand() < 0.5


### Example agents

In [5]:
class agentRandom(agent): # Yahan bracket mein (agent) likhne ka matlab hai ke agentRandom ek child class hai aur agent iski parent class hai. Child class parent ke saare functions ko khud-ba-khud inherit kar leti hai
    def __init__(self, name):
        super().__init__(name) # super() ka matlab hota hai "Parent Class". Yahan yeh code keh raha hai ke "Parent class ka __init__ method chalao aur usko yeh name de do". Is wajah se humein is class mein dobara self.name = name nahi likhna pada; parent class khud hi naam set kar degi

    def __call__(self, agent_history, opponent_history, burn_in=False):
        # Random strategy: cooperate with a probability of 0.5
        return np.random.rand() < 0.5

In [6]:
class agentCooperate(agent):
    def __init__(self, name):
        super().__init__(name)
        
    def __call__(self, agent_history, opponent_history, burn_in=False):
        return True  # Always cooperate

In [7]:
class agentDefect(agent):
    def __init__(self, name):
        super().__init__(name)

    def __call__(self, agent_history, opponent_history, burn_in=False):
        # Always defect
            return False

### Iterated prisoner's dilemma
* Agent input: own history, opponent history
* Agent return: True (cooperate) or False (defect)

Scoring:
* Both cooperate: 0 point each
* Both defect: 2 points each
* One cooperates, one defects: Cooperator: 5, Defector: 1
Lowest score wins

In [8]:
def prisoners_dilemma(agent_1, agent_2, rounds=10, burn_in=0):
    # Scoring system:
    # Both cooperate: Both get 0 point
    bothCooperate = 0
    # One cooperates, one defects: Cooperator gets 5 points, Defector gets 1 points
    oneCooperateOneDefect_Cooperate = 5
    oneCooperateOneDefect_Defect = 1
    # Both defect: Both get 2 point
    bothDefect = 2


    agent_1_history = [] # ki list mein unki pichli saari chaalein (moves) record hongi.
    agent_2_history = []
    agent_1_score = 0
    agent_1_score2 = 0   # squared score for stdv calculation / unke points ka square (murabba) jama karega, jo aakhir mein maths (Standard Deviation) ke kaam aayega
    agent_2_score = 0
    agent_2_score2 = 0   # squared score for stdv calculation / unke points ka square (murabba) jama karega, jo aakhir mein maths (Standard Deviation) ke kaam aayega

    # Burn-in phase: play a few rounds without scoring to establish history
    for _ in range(burn_in):
        agent_1_move = agent_1(agent_1_history, agent_2_history, True)
        agent_2_move = agent_2(agent_2_history, agent_1_history, True)
        
        agent_1_history.append(agent_1_move)
        agent_2_history.append(agent_2_move)


    # Actual game rounds
    for _ in range(rounds):
        agent_1_move = agent_1(agent_1_history, agent_2_history, False)
        agent_2_move = agent_2(agent_2_history, agent_1_history, False)
        
        if agent_1_move and agent_2_move:   # both cooperate
            agent_1_score += bothCooperate
            agent_2_score += bothCooperate
            agent_1_score2 += bothCooperate**2
            agent_2_score2 += bothCooperate**2
        elif agent_1_move and not agent_2_move:  # agent 1 cooperates, agent 2 defects
            agent_1_score += oneCooperateOneDefect_Cooperate
            agent_1_score2 += oneCooperateOneDefect_Cooperate**2
            agent_2_score += oneCooperateOneDefect_Defect
            agent_2_score2 += oneCooperateOneDefect_Defect**2
        elif not agent_1_move and agent_2_move:  # agent 1 defects, agent 2 cooperates
            agent_1_score += oneCooperateOneDefect_Defect
            agent_1_score2 += oneCooperateOneDefect_Defect**2
            agent_2_score += oneCooperateOneDefect_Cooperate
            agent_2_score2 += oneCooperateOneDefect_Cooperate**2
        else:  # both defect
            agent_1_score += bothDefect
            agent_2_score += bothDefect
            agent_1_score2 += bothDefect**2
            agent_2_score2 += bothDefect**2

        agent_1_history.append(agent_1_move)
        agent_2_history.append(agent_2_move)

    agent_1_stdv = np.sqrt((agent_1_score2 - agent_1_score**2/rounds)/(rounds-1))
    agent_2_stdv = np.sqrt((agent_2_score2 - agent_2_score**2/rounds)/(rounds-1))

    return agent_1_score/rounds, agent_2_score/rounds, agent_1_stdv, agent_2_stdv   

In [9]:
def round_robin_tournament(agents, rounds=10, burn_in=0):

        scores = np.zeros(len(agents))   # sum of scores
        stdvs = np.zeros(len(agents))    # sum of standard deviations

        # Randomize playing order of agents
        playing_order = np.arange(len(agents))
        playing_order = np.random.permutation(playing_order)

        # Round-robin tournament with burn-in period where each agent plays against every other agent for a specified number of rounds, but the first few rounds are not counted towards the final score
        for i in range(len(agents)):
            for j in range(i + 1, len(agents)):
                    a1 = agents[playing_order[i]]
                    a2 = agents[playing_order[j]]
                    agent_1_score, agent_2_score, agent_1_stdv, agent_2_stdv = prisoners_dilemma(a1, a2, rounds=rounds, burn_in=burn_in)
                    scores[playing_order[i]] += agent_1_score
                    stdvs[playing_order[i]] += agent_1_stdv
                    scores[playing_order[j]] += agent_2_score
                    stdvs[playing_order[j]] += agent_2_stdv

        scores /= (len(agents) - 1)       # average score per match
        stdv = stdvs / (len(agents) - 1)  # average standard deviation per match
        return scores, stdv

### Standard round-robin turnament

* All agents play each-other once 
* Each game consits of 25 iterations of prisoner's dilemma
* All iterations are scored

In [10]:
agents = [agentRandom('Random'), 
          agentCooperate('Cooperate'), 
          agentDefect('Defect')]

scores, stdv = round_robin_tournament(agents, rounds=25, burn_in=0)

# Print the final scores of each agent after the tournament
for i, score in enumerate(scores):
    print(f"Agent {agents[i].name} final score:\t\t {score:.1f}, (stdv: {stdv[i]:.1f})")

Agent Random final score:		 2.0, (stdv: 1.0)
Agent Cooperate final score:		 3.8, (stdv: 1.3)
Agent Defect final score:		 1.3, (stdv: 0.3)


### Round-robin turnament with burn-in phase

* All agents play each-other once 
* Each game starts with 500 unscored iterations of prisoner's dilemma, followed by 25 scored iterations

In [11]:
agents = [agentRandom('Random'), 
          agentCooperate('Cooperate'), 
          agentDefect('Defect')]

scores, stdv = round_robin_tournament(agents, rounds=25, burn_in=500)

# Print the final scores of each agent after the tournament
for i, score in enumerate(scores):
    print(f"Agent {agents[i].name} final score:\t\t {score:.1f}, (stdv: {stdv[i]:.1f})")

Agent Random final score:		 1.8, (stdv: 1.0)
Agent Cooperate final score:		 4.1, (stdv: 1.2)
Agent Defect final score:		 1.3, (stdv: 0.2)
